# 04 · Eleição do vencedor e promoção controlada

**Etapa 1 — vencedor do ciclo:** maior média de AUC-PR na validação cruzada (notebook 03). Empate: menor desvio-padrão.

**Etapa 2 — vencedor × vigente:** os dois são avaliados na **mesma validação do ciclo** (dados que nenhum dos dois viu no treino).

| Situação | Decisão |
|---|---|
| Candidato não supera o modelo aleatório (AUC-PR ≤ prevalência) | REJEITADO, vigente mantido |
| Não existe vigente | PROMOVIDO |
| Candidato ≥ vigente + 0,005 | PROMOVIDO |
| Caso contrário | MANTIDO |

O teste (15% reservado) só é usado no fim, para reportar o desempenho do modelo que ficou vigente. Qualquer erro neste notebook interrompe o workflow **antes** de o novo estado ser salvo, então nunca há promoção indevida.

In [1]:
import os, sys, warnings
from pathlib import Path
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore", category=UserWarning)
# Localiza a raiz do repositório e torna o pacote src importável
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
os.chdir(RAIZ); sys.path.insert(0, str(RAIZ))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from src.utils import carregar_config, configurar_mlflow, ler_contexto, atualizar_contexto, tags_ciclo, caminho, salvar_json
cfg = carregar_config()
pd.set_option("display.width", 160)

In [2]:
import joblib
from src.preprocessamento import separar_xy
from src.modelagem import calcular_metricas
from src.promocao import eleger_vencedor, decidir_promocao, carregar_vigente, promover, registrar_decisao, ler_manifesto
from src.utils import resumo_github

ctx = ler_contexto(cfg)
metrica, direcao = cfg["avaliacao"]["metrica_principal"], cfg["avaliacao"]["direcao"]
pct = cfg["negocio"]["percentual_contato"]
proc = caminho(cfg["dados"]["processed_dir"])
validacao = pd.read_csv(proc / "validacao.csv"); teste = pd.read_csv(proc / "teste.csv")
X_va, y_va = separar_xy(validacao, cfg); X_te, y_te = separar_xy(teste, cfg)
cv = pd.read_csv(caminho(cfg["dados"]["state_dir"]) / "cv_ciclo.csv")
print(ctx["cycle_id"], ctx["data_version"])

c001-20261116-2102 v001-44df464c


## 1. Vencedor do ciclo (A × B)

In [3]:
vencedor, ranking = eleger_vencedor(cv, metrica, direcao)
display(ranking.round(4))
cand = ctx["candidatos"][vencedor]
pipe_cand = joblib.load(caminho(cfg["dados"]["state_dir"]) / "candidatos" / f"{vencedor}.joblib")
print(f"Vencedor do ciclo: Modelo {vencedor} ({cand['algoritmo']})")

,modelo,algoritmo,average_precision_media,average_precision_desvio
0,B,random_forest,0.1004,0.0052
1,A,regressao_logistica,0.0902,0.0070


Vencedor do ciclo: Modelo B (random_forest)


## 2. Vencedor × modelo vigente (mesma validação)

In [4]:
vigente, manif_ant = carregar_vigente()   # vem do armazenamento persistente (repositório), não do runner
m_cand = calcular_metricas(y_va, pipe_cand.predict_proba(X_va)[:, 1], pct)
m_vig = calcular_metricas(y_va, vigente.predict_proba(X_va)[:, 1], pct) if vigente is not None else None

# Verificações de sanidade: falhas aqui interrompem o fluxo antes da promoção
assert len(ctx["features_eleitas"]) > 0, "Conjunto de features vazio"
assert np.isfinite(m_cand[metrica]), "Métrica do candidato inválida"
assert list(pipe_cand.named_steps["selecao"].features) == ctx["features_eleitas"], "Candidato não usa as features eleitas do ciclo"

piso = m_cand["prevalencia"]
decisao = decidir_promocao(m_cand[metrica], None if m_vig is None else m_vig[metrica], piso, cfg)

comp = pd.DataFrame({f"candidato ({cand['algoritmo']}, {ctx['cycle_id']})": m_cand,
                     **({f"vigente ({manif_ant['algoritmo']}, {manif_ant['cycle_id']})": m_vig} if m_vig else {})}).round(4)
display(comp)
display(Markdown(f"### Decisão: **{decisao['decisao']}**  \n{decisao['motivo']}"))

,"candidato (random_forest, c001-20261116-2102)"
average_precision,0.1045
roc_auc,0.6214
f1,0.1141
precision,0.0651
recall,0.4624
prevalencia,0.0426
conversao_top,0.0702
lift_top,1.6482
captura_top,0.3295


### Decisão: **PROMOVIDO**  
Não havia modelo vigente (primeiro ciclo).

## 3. Aplicação da decisão e manifesto

In [5]:
if decisao["decisao"] == "PROMOVIDO":
    manifesto = promover(pipe_cand, {
        "cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "modelo": vencedor,
        "algoritmo": cand["algoritmo"], "run_id": cand["run_id"], "features": ctx["features_eleitas"],
        "metrica_principal": metrica, "valor_validacao": m_cand[metrica],
        "cv_media": cand[f"cv_{metrica}_media"], "decisao": decisao,
        "drift_alerta_no_ciclo": ctx.get("drift_alerta"), "features_com_drift": ctx.get("features_com_drift"),
        "gatilho": ctx["execucao"]["gatilho"]})
    vigente = pipe_cand
else:
    manifesto = ler_manifesto()
    if manifesto is None:
        raise RuntimeError("Nenhum modelo vigente e o candidato foi rejeitado: pipeline interrompido.")
print(json.dumps(manifesto, indent=2, ensure_ascii=False))

{
  "cycle_id": "c001-20261116-2102",
  "data_version": "v001-44df464c",
  "modelo": "B",
  "algoritmo": "random_forest",
  "run_id": "62a8621e797144deabc85e1a9e4d1072",
  "features": [
    "euribor3m",
    "month",
    "education",
    "job",
    "age",
    "emp.var.rate",
    "contatos_totais",
    "day_of_week"
  ],
  "metrica_principal": "average_precision",
  "valor_validacao": 0.10449441252502639,
  "cv_media": 0.10039310126578052,
  "decisao": {
    "decisao": "PROMOVIDO",
    "motivo": "Não havia modelo vigente (primeiro ciclo)."
  },
  "drift_alerta_no_ciclo": false,
  "features_com_drift": [],
  "gatilho": "schedule",
  "arquivo_modelo": "models/historico/c001-20261116-2102_random_forest.joblib",
  "promovido_em": "2026-11-16T21:04:11+00:00",
  "substituiu": null
}


## 4. Avaliação final do modelo vigente no teste reservado

In [6]:
m_teste = calcular_metricas(y_te, vigente.predict_proba(X_te)[:, 1], pct)
display(pd.Series(m_teste, name="teste").round(4).to_frame())

mlflow = configurar_mlflow(cfg)
with mlflow.start_run(run_name=f"{ctx['cycle_id']}_04_decisao", tags={**tags_ciclo(ctx, "04_promocao"), "decisao": decisao["decisao"]}) as run:
    mlflow.log_params({"vencedor_ciclo": vencedor, "algoritmo_vencedor": cand["algoritmo"], "run_id_vencedor": cand["run_id"],
                       "decisao": decisao["decisao"], "motivo": decisao["motivo"][:250],
                       "limiar_melhoria": cfg["promocao"]["limiar_melhoria"],
                       "vigente_cycle_id": manifesto["cycle_id"], "vigente_run_id": manifesto["run_id"]})
    mlflow.log_metrics({f"candidato_val_{metrica}": m_cand[metrica], f"piso_aleatorio_{metrica}": piso,
                        **({f"vigente_anterior_val_{metrica}": m_vig[metrica]} if m_vig else {}),
                        **{f"vigente_teste_{k}": v for k, v in m_teste.items()}})
    mlflow.log_dict(manifesto, "modelo_vigente.json")
    mlflow.log_table(ranking, "ranking_vencedor.json")

linha = {"cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "gatilho": ctx["execucao"]["gatilho"],
         "lote": ctx["lote"].get("lote"), "lote_novo": ctx["lote"]["novo"], "linhas_lote": ctx["lote"]["linhas"],
         "features_eleitas": "|".join(ctx["features_eleitas"]), "n_features": len(ctx["features_eleitas"]),
         "cv_A": ctx["candidatos"]["A"][f"cv_{metrica}_media"], "cv_B": ctx["candidatos"]["B"][f"cv_{metrica}_media"],
         "vencedor": f"{vencedor}-{cand['algoritmo']}", f"candidato_val_{metrica}": m_cand[metrica],
         f"vigente_anterior_val_{metrica}": None if m_vig is None else m_vig[metrica],
         "decisao": decisao["decisao"], "vigente_apos": f"{manifesto['cycle_id']}-{manifesto['algoritmo']}",
         f"vigente_teste_{metrica}": m_teste[metrica], "vigente_teste_lift_top": m_teste["lift_top"],
         "drift_alerta": ctx.get("drift_alerta"), "psi_max": ctx.get("psi_max"), "run_decisao": run.info.run_id}
historico = registrar_decisao(linha)
display(historico.tail(5))

resumo_github(f"### Decisão: **{decisao['decisao']}**\n\n{decisao['motivo']}\n\n"
              f"| | valor |\n|---|---|\n| vencedor do ciclo | {vencedor} ({cand['algoritmo']}) |\n"
              f"| features | {', '.join(ctx['features_eleitas'])} |\n| vigente | {manifesto['cycle_id']} ({manifesto['algoritmo']}) |\n"
              f"| {metrica} no teste (vigente) | {m_teste[metrica]:.4f} |\n")
atualizar_contexto(cfg, decisao=decisao, run_decisao=run.info.run_id)

,teste
average_precision,0.0936
roc_auc,0.5708
f1,0.1176
precision,0.0691
recall,0.3935
prevalencia,0.0529
conversao_top,0.0735
lift_top,1.3903
captura_top,0.2778


,cycle_id,data_version,gatilho,lote,lote_novo,linhas_lote,features_eleitas,n_features,cv_A,cv_B,vencedor,candidato_val_average_precision,vigente_anterior_val_average_precision,decisao,vigente_apos,vigente_teste_average_precision,vigente_teste_lift_top,drift_alerta,psi_max,run_decisao
0,c001-20261116-2102,v001-44df464c,schedule,0,True,27680,euribor3m|month|education|job|age|emp.var.rate...,8,0.090226,0.100393,B-random_forest,0.104494,None,PROMOVIDO,c001-20261116-2102-random_forest,0.093592,1.390251,False,None,1456d4163be643ec93414675f1013b68


{'cycle_id': 'c001-20261116-2102',
 'data_version': 'v001-44df464c',
 'execucao': {'gatilho': 'schedule',
  'github_run_id': '43955692226',
  'github_run_url': 'https://github.com/weillerjulia/n3-mlops-bank-marketing/actions/runs/43955692226',
  'commit': 'aeb605a',
  'iniciado_em': '2026-11-16T21:02:46+00:00'},
 'lote': {'novo': True,
  'lote': 0,
  'arquivo': 'data/raw/lote_000.csv',
  'periodos': ['2008-05',
   '2008-06',
   '2008-07',
   '2008-08',
   '2008-10',
   '2008-11'],
  'linhas': 27680,
  'sha256': '9f2fcaccde47',
  'ingerido_em': '2026-11-16T21:02:46+00:00'},
 'drift_alerta': False,
 'features_com_drift': [],
 'psi_max': None,
 'run_etl': 'd3b603e6577b4d4b9587ba59c7d64a87',
 'features_eleitas': ['euribor3m',
  'month',
  'education',
  'job',
  'age',
  'emp.var.rate',
  'contatos_totais',
  'day_of_week'],
 'run_votacao': '3f50e8a7ff8e4ad385bfa397a0cdba14',
 'candidatos': {'A': {'run_id': '6d093a900ae742fe886a9d2ed7012c2c',
   'algoritmo': 'regressao_logistica',
   'val'